# 08 — Consolidated Experiment Record

This final notebook writes one self-contained, versionable JSON record at the repository root. It embeds the resolved configuration, hardware/software provenance, seeds, tokenizer and model metadata, MLM metrics, all downstream seed outputs, and paired statistical analysis. Running it overwrites `experiment_record.json` atomically. The same collector is called automatically by the final statistical-analysis pipeline stage.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def find_project_root() -> Path:
    configured_root = os.environ.get('PROJECT_ROOT')
    starting_points = [Path(configured_root)] if configured_root else [Path.cwd()]
    for starting_point in starting_points:
        resolved_start = starting_point.expanduser().resolve()
        for candidate in (resolved_start, *resolved_start.parents):
            if (candidate / 'configs' / 'config.yaml').is_file():
                return candidate
    raise FileNotFoundError('Launch Jupyter from the repository root or set PROJECT_ROOT.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'PROJECT_ROOT: {PROJECT_ROOT}')


## Write the consolidated record

The collector reads only persisted project artifacts. It never trains a model, changes tokenizer artifacts, or mutates source data. Missing artifacts are reported explicitly, which makes the notebook safe to use after a partial run as well.

In [ ]:
import json

import yaml

from src.evaluation.experiment_record import write_experiment_record

config = yaml.safe_load((PROJECT_ROOT / 'configs' / 'config.yaml').read_text(encoding='utf-8'))
record_path = write_experiment_record(PROJECT_ROOT, config)
record = json.loads(record_path.read_text(encoding='utf-8'))
print(json.dumps({
    'path': str(record_path),
    'experiment_id': record['experiment']['id'],
    'training_seed': record['experiment']['training_seed'],
    'evaluation_seeds': record['experiment']['evaluation_seeds'],
    'missing_artifacts': record['missing_artifacts'],
}, indent=2))


## Verify version-control status

`experiment_record.json` is deliberately outside ignored artifact folders and explicitly unignored in `.gitignore`, so it can be committed with source and configuration changes.

In [ ]:
import subprocess

check = subprocess.run(
    ['git', 'check-ignore', '-q', str(record_path)],
    cwd=PROJECT_ROOT,
    check=False,
)
if check.returncode == 0:
    raise AssertionError(f'{record_path.name} is ignored; it must remain versionable.')
if check.returncode not in {1}:
    raise RuntimeError(f'git check-ignore failed with return code {check.returncode}.')
print(f'{record_path.name} is not ignored and is ready to add to Git.')
